# COM-490 Final — Probabilistic Journey Planner

**Robust public-transport routing with real-time delay confidence for the Lausanne region.**

This notebook is the **user interface**. All heavy computation lives in `backend.py`.

---
### Workflow
| Step | What happens | Run once? |
|------|-------------|----------|
| 0 | Load environment & start Spark | Every session |
| 1 | Connect to Trino & prepare stop views | **Once per region set** |
| 2 | Build delay-distribution tables | **Once** (cached to HDFS) |
| 3 | Set journey parameters | Every query |
| 4 | Find & score candidate routes | Every query |
| 5 | Visualise results | Every query |

## Step 0 — Environment & Spark

In [1]:
%load_ext autoreload
%autoreload 2

from backend import (
    load_environment, create_spark_session,
    init_journey_planner, search_stops,
    build_delay_tables, load_trip_to_line,
    build_bus_snapshot, make_bus_state_fn,
    load_lookups, find_routes, score_routes,
    print_route_summary, print_breakdown, launch_ui,
    build_istdaten_clean, get_holdout_window
)

# Load & validate environment variables
cfg = load_environment()

print(f"username   = {cfg['username']}")
print(f"hadoopFS   = {cfg['hadoopFS']}")
print(f"USER_NS    = {cfg['USER_NS']}")
print(f"GROUP_NAME = {cfg['GROUP_NAME']}")
print(f"FINAL_DIR  = {cfg['FINAL_DIR']}")


username   = chaurasi
hadoopFS   = hdfs://iccluster061.iccluster.epfl.ch:9000
USER_NS    = iceberg.chaurasi_iceberg
GROUP_NAME = D2
FINAL_DIR  = hdfs://iccluster061.iccluster.epfl.ch:9000/user/chaurasi/final


In [2]:
# Start Spark — takes ~30 s on first run
spark = create_spark_session(cfg['username'], cfg['hadoopFS'])
print('Spark ready:', spark.sparkContext.applicationId)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/29 17:33:41 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
26/05/29 17:33:49 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Attempted to request executors before the AM has registered!


Spark ready: application_1777877431184_2344


---
## Step 1 — Connect to Trino & prepare stop views

> **Run `jp.prepare()` only once per region set.**  
> On subsequent sessions set `REGION_UUIDS = []` to skip it.

In [3]:
# ── Configure region UUIDs ────────────────────────────────────────────────
# Leave list non-empty on first run; set to [] once views are materialised.
REGION_UUIDS = [
    'a7a21b73-6ffe-4fbf-a635-6e2b961f3072',  # Lausanne region
    'e168fd57-f57a-4075-a350-0dcfbb55147f',  # Ouest Lausanne region
]

jp = init_journey_planner(
    USER_NS      = cfg['USER_NS'],
    REGION_UUIDS = REGION_UUIDS,
    force_prepare = False,  # Set True to force re-materialisation
)

Trino connection live.
Shared namespace : iceberg.com490_iceberg
User namespace   : iceberg.chaurasi_iceberg


/home/chaurasi/final/com490/journey_planner.py:137: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  max_pub_date = pd.read_sql(f"""


Max publication date: 2026-01-31
Finished Creating stops..
Finished Creating stops times..
Finished Creating walk stat times..


/home/chaurasi/final/com490/journey_planner.py:75: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  stops_df = pd.read_sql(f"SELECT * FROM {self.userns}.stops", self.conn)



Stops in iceberg.chaurasi_iceberg.stops: 390 distinct names.


In [4]:
# ── Explore available stop names ──────────────────────────────────────────
# Use this cell to find the exact stop names before setting START_NAME / END_NAME below.

for hit in search_stops(jp, 'EPFL'):
    print(f'  {hit[0]}  ->  {hit[1]}')

  Ecublens VD, EPFL  ->  8501214
  Ecublens VD, EPFL (bus)  ->  8579233
  Ecublens VD, EPFL Piccard  ->  8579234
  Ecublens VD, EPFL/Colladon  ->  8510727
  Lausanne, sports UNIL-EPFL  ->  8595937


In [5]:
for hit in search_stops(jp, 'Lausanne')[:15]:
    print(f'  {hit[0]}  ->  {hit[1]}')

  Lausanne  ->  8501120
  Lausanne, Abeilles  ->  8591977
  Lausanne, Allières/CHUV  ->  8591978
  Lausanne, Alpes  ->  8591979
  Lausanne, Aubépines  ->  8591980
  Lausanne, Barre  ->  8591982
  Lausanne, Batelière  ->  8591983
  Lausanne, Beau-Rivage  ->  8591986
  Lausanne, Beau-Site  ->  8591987
  Lausanne, Beaulieu  ->  8591984
  Lausanne, Beaulieu-Jomini  ->  8592060
  Lausanne, Bel-Air  ->  8591988
  Lausanne, Bellerive  ->  8591989
  Lausanne, Bellevaux  ->  8587357
  Lausanne, Belvédère  ->  8591990


In [6]:
# ── Build istdaten clean parquet (region-scoped, with holdout) ──────────
# Run once per region set. Holds out the last 30 days to prevent leakage
# when evaluating with a recent OPERATING_DAY.
build_istdaten_clean(spark, jp, cfg, holdout_days=30)

/home/chaurasi/final/com490/backend.py:245: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  stops_df = pd.read_sql(


Region scope: 390 distinct bpuic from iceberg.chaurasi_iceberg.stops


Available istdaten range up to : 2026-01-31
Training cutoff (max − 30d) : 2026-01-01
Holdout window for evaluation  : (2026-01-01, 2026-01-31]


Wrote hdfs://iccluster061.iccluster.epfl.ch:9000/user/chaurasi/final/istdaten_clean.parquet


---
## Step 2 — Build delay-distribution tables
Set `STOP_DELAY_DISTRIBUTION` to either
* `'gumbel'` (parametric Gumbel(μ, β))
* `'discrete'` (71-bucket PMF)

> **Run once.** Takes several minutes. Subsequent calls are instant (HDFS cache).

In [7]:
STOP_DELAY_DISTRIBUTION = 'discrete'   # or 'gumbel'

In [8]:
build_delay_tables(spark, cfg, stop_delay_distribution=STOP_DELAY_DISTRIBUTION)

Building stop_delays (discrete PMF)… (this takes a few minutes)


26/05/29 17:35:15 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


  stop_delays done.
Building segment_tt… (this takes a few minutes)


  segment_tt done.


---
## Step 3 — Journey parameters

**Edit the cells below for each new query.**

In [9]:
# ── Origin & destination ──────────────────────────────────────────────────
START_NAME = 'Ecublens VD, EPFL'  # exact name from search_stops() above
END_NAME   = 'Lausanne'           # exact name from search_stops() above

START_ID = jp.name_to_id[START_NAME]
END_ID   = jp.name_to_id[END_NAME]

print(f'Origin      : {START_NAME}  ({START_ID})')
print(f'Destination : {END_NAME}  ({END_ID})')


Origin      : Ecublens VD, EPFL  (8501214)
Destination : Lausanne  (8501120)


In [10]:
# Valid OPERATING_DAY inside the Holdout window
cutoff, max_day = get_holdout_window(spark, holdout_days=30)
print(f"Holdout window for OPERATING_DAY: ({cutoff}, {max_day}]")

Holdout window for OPERATING_DAY: (2026-01-01, 2026-01-31]


In [11]:
# ── Temporal parameters ───────────────────────────────────────────────────
DAY           = 'monday'     # lowercase weekday name
QUERY_TIME_S  = '08:30'      # time of the user query (HH:MM)
DEPARTS       = '08:30'      # earliest acceptable departure (HH:MM)
DEADLINE      = '09:30'      # latest acceptable arrival  (HH:MM)
MAX_WALK_M    = 500          # maximum walking distance in metres
Q_THRESHOLD   = 0.80         # minimum on-time confidence to consider a route "feasible"

# Operating day for real-time bus snapshot (must be in the Holdout window to avoid data leakage)
OPERATING_DAY = '2026-01-15'

# ── Derived constants (do not edit) ───────────────────────────────────────
from routing.time_utils import time_to_seconds

QUERY_TIME_S = time_to_seconds(QUERY_TIME_S)
DEADLINE_S   = time_to_seconds(DEADLINE)

print(f'Day          : {DAY}')
print(f'Departs      : {DEPARTS}  (query time = {QUERY_TIME_S//3600:02d}:{(QUERY_TIME_S%3600)//60:02d})')
print(f'Deadline     : {DEADLINE}')
print(f'Max walk     : {MAX_WALK_M} m')
print(f'Q threshold  : {Q_THRESHOLD:.0%}')
print(f'Operating day: {OPERATING_DAY}')

Day          : monday
Departs      : 08:30  (query time = 08:30)
Deadline     : 09:30
Max walk     : 500 m
Q threshold  : 80%
Operating day: 2026-01-15


---
## Step 4 — Load runtime ingredients

In [12]:
# trip_id → line name mapping (built via Trino, cached locally)
trip_to_line = load_trip_to_line(jp, cfg)

/home/chaurasi/final/com490/backend.py:378: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  max_pub = pd.read_sql(
/home/chaurasi/final/com490/backend.py:388: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, jp.conn)


trip_to_line built from Trino (1,409,571 rows, pub_date=2026-01-31).
trip_to_line: 1,409,571 rows loaded.


In [13]:
# Real-time bus snapshot (where is each bus right now?)
snapshot     = build_bus_snapshot(spark, cfg, OPERATING_DAY, QUERY_TIME_S, jp)
bus_state_fn = make_bus_state_fn(snapshot)

/home/chaurasi/final/com490/backend.py:433: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  stops_df = pd.read_sql(


Bus snapshot built and written to HDFS.
Snapshot rows: 1,573


In [14]:
# Delay & travel-time lookup objects (fetched from HDFS, cached locally)
stop_delays, travel_times = load_lookups(cfg, stop_delay_distribution=STOP_DELAY_DISTRIBUTION,)

Lookups loaded (stop_delay_distribution='discrete').


---
## Step 5 — Find candidate routes

In [15]:
candidates = find_routes(
    jp          = jp,
    start_id    = START_ID,
    end_id      = END_ID,
    day         = DAY,
    departs     = DEPARTS,
    deadline    = DEADLINE,
    max_walk_m  = MAX_WALK_M,
)

# Preview route signatures — note *w to handle extra event fields
from routing.time_utils import seconds_to_time
for depart_str, evs in candidates:
    sig = tuple(e[2] for e in evs if e[2] is not None)
    print(f'  {depart_str} -> {len(evs)} events, trips={sig}')
    for ev in evs:
        ts, x, y, z, *w = ev
        h, m = divmod(ts // 60, 60)
        s    = ts % 60
        if x is None and y is not None and z is not None:
            kind = 'ALIGHT'
            info = f'trip={y}  at stop={z}'
        elif x is not None and y is not None and z is None:
            kind = 'BOARD '
            info = f'trip={y}  at stop={x}'
        else:
            kind = 'WALK  '
            info = f'{x} -> {z}'
        print(f'    {h:02d}:{m:02d}:{s:02d}  {kind}  {info}')


Found 11 distinct candidate path(s).
  09:15 -> 6 events, trips=('2600.TA.91-m1-j26-1.7.R', '2600.TA.91-m1-j26-1.7.R', '1698.TA.91-33-C-j26-1.94.R', '1698.TA.91-33-C-j26-1.94.R')
    09:15:00  WALK    8501214 -> 8501215
    09:15:00  BOARD   trip=2600.TA.91-m1-j26-1.7.R  at stop=8501215
    09:19:00  ALIGHT  trip=2600.TA.91-m1-j26-1.7.R  at stop=8530749
    09:24:00  WALK    8530749 -> 8501118
    09:24:00  BOARD   trip=1698.TA.91-33-C-j26-1.94.R  at stop=8501118
    09:29:00  ALIGHT  trip=1698.TA.91-33-C-j26-1.94.R  at stop=8501120
  09:13 -> 5 events, trips=('2600.TA.91-m1-j26-1.7.R', '2600.TA.91-m1-j26-1.7.R', '1698.TA.91-33-C-j26-1.94.R', '1698.TA.91-33-C-j26-1.94.R')
    09:13:00  BOARD   trip=2600.TA.91-m1-j26-1.7.R  at stop=8501214
    09:19:00  ALIGHT  trip=2600.TA.91-m1-j26-1.7.R  at stop=8530749
    09:24:00  WALK    8530749 -> 8501118
    09:24:00  BOARD   trip=1698.TA.91-33-C-j26-1.94.R  at stop=8501118
    09:29:00  ALIGHT  trip=1698.TA.91-33-C-j26-1.94.R  at stop=8501120


---
## Step 6 — Score every candidate & rank

In [16]:
summary = score_routes(
    candidates   = candidates,
    stop_delays  = stop_delays,
    travel_times = travel_times,
    trip_to_line = trip_to_line,
    day          = DAY,
    query_time_s = QUERY_TIME_S,
    deadline_s   = DEADLINE_S,
    bus_state_fn = bus_state_fn,
    q_threshold  = Q_THRESHOLD,
)

print_route_summary(summary, DEADLINE, Q_THRESHOLD)


  Q threshold: 80%   deadline: 09:30
depart_str arr_str  n_transfers  overall_p  meets_Q
     09:15   09:29            1   0.784677    False
     09:13   09:29            1   0.784677    False
     09:10   09:26            1   0.965596     True
     09:10   09:29            1   0.795322    False
     09:08   09:29            1   0.626728    False
     09:08   09:29            2   0.571544    False
     09:08   09:29            2   0.571544    False
     09:08   09:26            1   0.976824     True
     09:08   09:26            1   0.961122     True
     09:08   09:29            1   0.795322    False
     09:08   09:26            1   0.976824     True

Feasible routes (4 of 11, sorted by latest departure):
depart_str arr_str  n_transfers  overall_p  meets_Q
     09:10   09:26            1   0.965596     True
     09:08   09:26            1   0.976824     True
     09:08   09:26            1   0.961122     True
     09:08   09:26            1   0.976824     True


In [17]:
# Detailed per-segment breakdown
print_breakdown(summary)


=== 09:15  total p=0.785 ===
  transfer @ 8530749: m1 → RE33, gap=300s, walk=32s, p=0.987
  arrival  @ 8501120: line RE33, cushion=60s, p=0.795

=== 09:13  total p=0.785 ===
  transfer @ 8530749: m1 → RE33, gap=300s, walk=32s, p=0.987
  arrival  @ 8501120: line RE33, cushion=60s, p=0.795

=== 09:10  total p=0.966 ===
  transfer @ 8530749: m1 → IC5, gap=300s, walk=32s, p=0.987
  arrival  @ 8501120: line IC5, cushion=240s, p=0.979

=== 09:10  total p=0.795 ===
  transfer @ 8530749: m1 → RE33, gap=540s, walk=32s, p=1.000
  arrival  @ 8501120: line RE33, cushion=60s, p=0.795

=== 09:08  total p=0.627 ===
  transfer @ 8501218: m1 → RE33, gap=720s, walk=499s, p=0.788
  arrival  @ 8501120: line RE33, cushion=60s, p=0.795

=== 09:08  total p=0.572 ===
  transfer @ 8501218: m1 → 33, gap=360s, walk=192s, p=0.734
  transfer @ 8511875: 33 → RE33, gap=300s, walk=110s, p=0.979
  arrival  @ 8501120: line RE33, cushion=60s, p=0.795

=== 09:08  total p=0.572 ===
  transfer @ 8501218: m1 → 33, gap=360s

---
## Step 7 — Interactive visualisation

In [18]:
launch_ui(
    jp               = jp,
    summary          = summary,
    candidates       = candidates,
    stop_delays      = stop_delays,
    travel_times     = travel_times,
    trip_to_line     = trip_to_line,
    day              = DAY,
    query_time_s     = QUERY_TIME_S,
    deadline_s       = DEADLINE_S,
    bus_state_fn     = bus_state_fn,
    q_threshold      = Q_THRESHOLD,
    origin_name      = START_NAME,
    destination_name = END_NAME,
    deadline         = DEADLINE,
)


/home/chaurasi/final/com490/backend.py:785: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  stops_df = pd.read_sql(f"SELECT * FROM {jp.userns}.stops", jp.conn)


---
## Cleanup

In [19]:
# Uncomment to stop the Spark session when you are done.
# spark.stop()